# Critérios de Divisibilidade e Otimização Bitwise
1. A Base MatemáticaNa teoria dos números, dizemos que um número inteiro $a$ divide um número <br>
inteiro $b$ (representado por $a \mid b$) se existe um inteiro $k$ tal que: $b = k.a$ <br>
Isso significa que a divisão de $b$ por $a$ deixa resto zero ($r = 0$).


2. A Visão do Hardware (O Pulo do Gato Binário)
Quando levamos isso para o processador, a base não é 10, é base 2 (binário). No binário, a mágica<br>
da divisibilidade por potências de 2 ($2^k$) se torna perfeita e instantânea.


Em binário, qualquer potência de 2 tem apenas um bit ligado (1) seguido de vários zeros. Por exemplo:
- $2^1 = 2 \rightarrow \text{binário } 10_2$
- $2^2 = 4 \rightarrow \text{binário } 100_2$
- $2^3 = 8 \rightarrow \text{binário } 1000_2$
- $2^4 = 16 \rightarrow \text{binário } 10000_2$

Se você quer saber se um número é divisível por $2^k$, você não precisa fazer a divisão real. Basta olhar <br>
para os últimos $k$ bits desse número. Se eles forem todos zeros, o número é divisível.

3. Traduzindo em Código de Baixo Nível (C)
Na arquitetura de um computador, uma instrução de divisão inteira na CPU (como a instrução IDIV no x86) pode <br>
custar de 15 a 40 ciclos de clock, sendo uma das operações aritméticas mais lentas.

Em contrapartida, uma operação lógica de Bitwise AND (&) custa apenas 1 ciclo de clock.

Podemos substituir a divisão por qualquer potência de 2 por uma máscara bitwise subtraindo 1 da potência:
- Testar se $n$ é divisível por 2: n & 1
- Testar se $n$ é divisível por 4: n & 3  (pois $4 - 1 = 3$, binário 011)i
- Testar se $n$ é divisível por 8: n & 7  (pois $8 - 1 = 7$, binário 111)i
- Testar se $n$ é divisível por $2^k$: n & ((1 << k) - 1)

```c
#include <stdio.h>
#include <stdbool.h>

// Otimização: Divisibilidade por potências de 2 usando Bitwise AND
bool eh_divisivel_por_potencia_de_2(int n, int k) {
    // k representa o expoente (ex: k=3 para testar divisão por 2^3 = 8)
    int mascara = (1 << k) - 1;
    
    // Se o resultado da máscara for 0, os últimos k bits eram todos 0
    return (n & mascara) == 0;
}

int main() {
    int numero = 40; // 40 em binário é 101000
    
    // Testando divisibilidade por 8 (2^3 -> k=3)
    if (eh_divisivel_por_potencia_de_2(numero, 3)) {
        printf("O numero %d é divisivel por 8 (Otimizado via Bitwise!)\n", numero);
    } else {
        printf("O numero %d NAO é divisivel por 8\n", numero);
    }
    
    return 0;
}

```
### O que aconteceu por trás dos panos?
O número 40 em binário é 00101000. <br>
O divisor é $8$ ($2^3$), então a nossa máscara é $8 - 1 = 7$ (que em binário é 00000111). <br>
Quando fazemos 40 & 7:
```
00101000 (40)
& 00000111 (7)
----------
  00000000 (0) -> Resto zero! Divisão exata sem usar a instrução de divisão da CPU.
```

### Panorama Geral
A Aula 1 estabelece a ponte fundamental entre a Teoria dos Números e o silício da CPU, demonstrando como os critérios de divisibilidade por potências de dois podem ser resolvidos instantaneamente através de operações lógicas de máscara com bitwise AND em vez de recorrer à custosa instrução de divisão inteira (IDIV). Esse panorama é complementado pela compreensão de como o sistema operacional e o compilador traduzem entradas baseadas em texto (como os códigos ASCII digitados no teclado) em valores binários nativos armazenados em registradores, revelando que a verdadeira performance de baixo nível exige tanto a eliminação de ciclos ociosos de hardware quanto a eliminação de conversões desnecessárias em tempo de execução.

| Instrução Assembly | Descrição da Operação | Latência Típica (Ciclos de Clock) | Observações de Hardware |
| :--- | :--- | :--- | :--- |
| **`NOP` / `MOV` (reg-reg)** | Nenhuma / Cópia entre registradores | **0 a 1** | Resolvidas por renomeação de registradores (custo zero real). |
| **`AND`, `OR`, `XOR`** | Operações lógicas (Máscaras de bits) | **0.5 a 1** | Executadas em paralelo em múltiplos *ports* da ALU. |
| **`ADD`, `SUB`** | Soma e Subtração inteira | **1** | A base fundamental dos cálculos na CPU. |
| **`SHL`, `SHR`** | Deslocamento de bits (*Shift*) | **1** | Multiplicação ou divisão instantânea por potências de 2. |
| **`IMUL`** | Multiplicação inteira | **3 a 4** | Processada em circuitos combinacionais rápidos (árvores de Wallace). |
| **`IDIV`** | Divisão inteira (com sinal) | **15 a 40+** | **O gargalo clássico:** processo iterativo sequencial que trava o pipeline. |
| **`MOV` (L1 Cache)** | Leitura de dado da Cache L1 | **~4** | (Se houver *cache miss* para a RAM principal, sobe para 200+ ciclos). |

| Flag | Nome Completo | Quando Vira **1** (Ativada) | Quando Vira **0** (Desativada) | Operações que Afetam / Notas |
| :--- | :--- | :--- | :--- | :--- |
| **CF** | *Carry Flag* (Vai-um / Empréstimo) | Houve *carry* na saída do bit mais alto (soma sem sinal) ou *borrow* (subtração em que $A < B$). | Não houve *carry* ou empréstimo na operação. | Afetada por `ADD`, `SUB`, `CMP`, deslocamentos (`SHL`, `SHR`). *Ignorada por `INC`/`DEC`.* |
| **PF** | *Parity Flag* (Paridade) | O byte menos significativo do resultado possui um **número par** de bits `1`. | O byte menos significativo possui um **número ímpar** de bits `1`. | Afetada pela maioria das instruções aritméticas e lógicas. Uso histórico para checagem de erros de transmissão. |
| **AF** | *Auxiliary Carry* (Carry Auxiliar) | Houve vai-um (*carry*) ou empréstimo (*borrow*) especificamente entre o bit 3 e o bit 4 (meio do byte). | Não houve vai-um/empréstimo entre o nibble inferior e o superior. | Vital para ajustes de aritmética decimal compactada (**BCD**). |
| **ZF** | *Zero Flag* (Zero) | O resultado matemático da operação foi **exatamente 0**. | O resultado da operação foi **diferente de 0**. | O termômetro de igualdade. Se `sub A, B` resulta em zero, `ZF = 1` ($A = B$). |
| **SF** | *Sign Flag* (Sinal) | O bit mais significativo do resultado é **1** (indicando número negativo em complemento de dois). | O bit mais significativo do resultado é **0** (indicando número positivo ou zero). | Copia diretamente o bit de sinal do resultado (bit 7 em 8 bits, bit 31 em 32 bits). |
| **OF** | *Overflow Flag* (Estouro com Sinal) | Ocorreu estouro na faixa de números com sinal (ex: somar dois números positivos e o resultado estourar o limite, virando negativo). | O resultado coube perfeitamente na faixa de representação com sinal. | Crítica para aritmética com sinal (`IMUL`, `ADD`, `SUB`). |